# Full Pipeline: 1500+ Videos Prosody Extraction + Training

**Goal:** Process 1500+ videos for laughter detection

**Steps:**
1. Mount Drive and extract existing audio
2. Download Gillick 988 (pre-labeled) + more comedy
3. Extract 23-dim prosody for all
4. Combine with existing 87 Gillick + 481 YouTube
5. Train fusion model

**Expected runtime:** 8-12 hours (GPU extraction)

In [ ]:
# @title Step 1: Setup
!pip install -q librosa scikit-learn
import os
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')
print('GPU:', !nvidia-smi --query-gpu=name --format=csv,noheader)

In [ ]:
# @title Step 2: Extract existing audio archive
import subprocess

# Extract existing tar.gz
print('Extracting vtt_audio_local.tar.gz...')
result = subprocess.run([
    'tar', '-xzf', '/content/gdrive/MyDrive/vtt_audio_local.tar.gz',
    '-C', '/content/'
], capture_output=True, text=True)
print('Return:', result.returncode)

# Count extracted files
extracted = subprocess.run(['find', '/content/vtt_audio_local', '-name', '*.m4a'], 
                         capture_output=True, text=True)
n_files = len(extracted.stdout.strip().split('\n'))
print(f'Extracted: {n_files} audio files')

In [ ]:
# @title Step 3: Download Gillick 988 videos
# Read Gillick video IDs from Drive
with open('/content/gdrive/MyDrive/gillick_988_videos.txt') as f:
    gillick_ids = [line.strip() for line in f if line.strip()]
print(f'Gillick videos to download: {len(gillick_ids)}')

# Download with yt-dlp (CPU, parallel)
import subprocess
from concurrent.futures import ThreadPoolExecutor

def download_video(vid):
    out_path = f'/content/gillick_audio/{vid}.mp3'
    if os.path.exists(out_path):
        return vid, True
    os.makedirs('/content/gillick_audio', exist_ok=True)
    r = subprocess.run([
        'yt-dlp', '--extract-audio', '--audio-format', 'mp3',
        '--audio-quality', '5', '-o', out_path,
        f'https://youtu.be/{vid}'
    ], capture_output=True, timeout=300)
    return vid, r.returncode == 0

# Download in parallel (10 at a time)
print('Downloading Gillick videos...')
with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(download_video, gillick_ids[:100]))  # Start with 100

success = sum(1 for _, ok in results if ok)
print(f'Downloaded: {success}/{len(results)}')

In [ ]:
# @title Step 4: Extract prosody for all videos
import numpy as np
import librosa
from tqdm import tqdm

SR = 16000

def extract_prosody_23dim(y, sr):
    features = []
    
    # F0 - 5 dims
    try:
        f0, voiced, probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([np.mean(f0_clean), np.std(f0_clean),
                        np.max(f0_clean), np.min(f0_clean),
                        np.sum(voiced)/len(voiced)])
    except:
        features.extend([0]*5)
    
    # Energy - 5 dims
    rms = librosa.feature.rms(y=y)[0]
    features.extend([np.mean(rms), np.std(rms), np.max(rms),
                    np.min(rms), np.max(rms)-np.min(rms)])
    
    # Duration - 2 dims
    features.extend([len(y)/sr, len(y)/sr/(np.sum(rms>np.mean(rms))+1])
    
    # Spectral - 5 dims
    try:
        spec = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        spec_flat = librosa.feature.spectral_flatness(y=y)[0]
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        features.extend([np.mean(spec), np.mean(spec_bw),
                        np.mean(spec_flat), np.mean(zcr), np.std(zcr)])
    except:
        features.extend([0]*5)
    
    # Voice quality - 4 dims
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr)/(np.mean(np.abs(y))+1e-8)
    except:
        hnr_val = 0
    features.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    
    # Delta - 2 dims
    try:
        delta_rms = np.diff(rms)
        features.extend([np.mean(np.abs(delta_rms)), np.max(np.abs(delta_rms))])
    except:
        features.extend([0]*2)
    
    return np.array(features, dtype=np.float32)

def process_video(audio_path, segments):
    y, sr = librosa.load(audio_path, sr=SR, mono=True)
    prosody = []
    for seg in segments:
        start, end = int(seg['start']*SR), int(seg['end']*SR)
        y_seg = y[start:end] if end <= len(y) else y[start:]
        if len(y_seg) < SR * 0.1:
            prosody.append(np.zeros(23, dtype=np.float32))
        else:
            prosody.append(extract_prosody_23dim(y_seg, SR))
    return np.array(prosody)

# Process all videos
BASE = Path('/content/gdrive/MyDrive/')
AUDIO_DIRS = ['vtt_audio_local', 'gillick_audio']

# Get all audio files
all_audio = {}
for d in AUDIO_DIRS:
    audio_dir = Path(f'/content/{d}')
    if audio_dir.exists():
        for f in audio_dir.glob('*.m4a'):
            all_audio[f.stem] = str(f)
        for f in audio_dir.glob('*.mp3'):
            all_audio[f.stem] = str(f)

print(f'Found {len(all_audio)} audio files')

# Load utterances for labels
import json
utt_path = BASE / 'utterances_clean.jsonl'
if not utt_path.exists():
    # Download from Drive
    subprocess.run(['gdown', '1cuhs6mh-r9Spzq9cTDG8AidT53DLsALn', 
                    '-O', str(utt_path)], check=True)

utterances = {}
with open(utt_path) as f:
    for line in f:
        d = json.loads(line)
        vid = d['video_id']
        if vid not in utterances:
            utterances[vid] = []
        utterances[vid].append(d)

print(f'Loaded {len(utterances)} videos with utterances')

# Process
all_prosody = {}
for vid, audio_path in tqdm(all_audio.items()):
    if vid in utterances:
        segs = [{'start': u['start'], 'end': u['end']} for u in utterances[vid]]
        labels = [u.get('label', 0) for u in utterances[vid]]
        prosody = process_video(audio_path, segs)
        all_prosody[vid] = {'prosody': prosody, 'labels': np.array(labels)}

print(f'Processed {len(all_prosody)} videos')

In [ ]:
# @title Step 5: Combine with existing data and train
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# Load existing 87 Gillick data
existing = np.load('/content/gdrive/MyDrive/wavlm_training_data_expanded.npz',
                    allow_pickle=True)
X_ex = existing['prosody']  # (21468, 23)
y_ex = existing['labels']

# Combine with new data
X_new = []
y_new = []
for vid, data in all_prosody.items():
    X_new.append(data['prosody'])
    y_new.append(data['labels'])

X_new = np.vstack(X_new)
y_new = np.concatenate(y_new)

# Combine
X_all = np.vstack([X_ex, X_new])
y_all = np.concatenate([y_ex, y_new])

print(f'Total: {len(y_all)} utterances')
print(f'Positive: {y_all.sum()} ({100*y_all.mean():.1f}%)')

# Train-val split by video (avoid leakage)
# ... (simplified here)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

# MLP Model
class LaughterMLP(torch.nn.Module):
    def __init__(self, dim=23):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(dim, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(128, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

# Train
model = LaughterMLP().cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3.0]).cuda())

train_ds = torch.utils.data.TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32))
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

for epoch in range(20):
    model.train()
    for x, y in train_loader:
        x, y = x.cuda(), y.cuda()
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    
    # Eval
    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(torch.tensor(X_test).cuda())) > 0.5
    f1 = f1_score(y_test, preds.cpu())
    print(f'Epoch {epoch+1}: F1={f1:.4f}')

In [ ]:
# @title Step 6: Save results to Drive
# Save trained model
torch.save(model.state_dict(), '/content/gdrive/MyDrive/prosody_mlp_1500.pt')
print('Model saved!')

# Save combined dataset
np.savez_compressed('/content/gdrive/MyDrive/prosody_1500_dataset.npz',
                  X=X_all, y=y_all)
print('Dataset saved!')